# P3-2 · Bài 3 · Attention và Transformer

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch, math
def attention(q, k, v, mask=None):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))   # (..., T, T)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    return torch.softmax(scores, dim=-1) @ v                    # (..., T, d_k)

In [ ]:
T = 8
mask = torch.tril(torch.ones(T, T))     # tam giác dưới: 1 = được nhìn, 0 = bị che

In [ ]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h = n_heads
        self.qkv = torch.nn.Linear(d_model, 3 * d_model)      # W_Q, W_K, W_V gộp lại
        self.o = torch.nn.Linear(d_model, d_model)            # W_O

    def forward(self, x, causal=False):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)                                   # mỗi cái (B, T, D)
        q, k, v = [t.view(B, T, self.h, D // self.h).transpose(1, 2) for t in (q, k, v)]   # (B, h, T, d_k)
        att = q @ k.transpose(-2, -1) / math.sqrt(D // self.h)                    # (B, h, T, T)
        if causal:
            att = att.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool), 1), float("-inf"))
        y = torch.softmax(att, dim=-1) @ v                                        # (B, h, T, d_k)
        return self.o(y.transpose(1, 2).reshape(B, T, D))                         # (B, T, D)